# Negotiation course: weekly sales before and after

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Paired Test for Means** · a worked example from the StatNex teaching guide

**The story.** Seven sales reps attended a one-day negotiation course. Their weekly sales (\$000) were recorded the week before and the week after.

**The question.** Did the reps sell more after the training?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/paired_mean_test.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Rep,Before training,After training
Alex,12.4,13.9
Bea,10.8,11.6
Chen,15.1,16.8
Dana,9.7,9.5
Eli,13.3,14.6
Farah,11.2,12.9
Gus,14.0,14.4

These cells save `cohen_d_interval.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile cohen_d_interval.py
"""Cohen's d with its noncentral-t interval, and Hedges' exact small-sample correction.

SciPy has the noncentral t distribution (stats.nct) but no function that inverts it for
an interval, so the inversion is written out (Steiger & Fouladi 1997). The interval is
for δ = t * scale: 1/sqrt(n) for one sample or paired differences, sqrt(1/n1 + 1/n2) for
two groups with the pooled standard deviation.
"""

import math

import numpy as np
from scipy import optimize, special, stats


def _limit(t, df, q):
    """The noncentrality under which the observed t sits at the q quantile of t(df, ncp).

    The cdf at t falls as the noncentrality rises, so the bracket opens outwards from t,
    doubling, until it holds the root.
    """
    def gap(ncp):
        return stats.nct.cdf(t, df, ncp) - q

    step = 1.0
    while gap(t - step) <= 0:
        step *= 2
    low = t - step
    step = 1.0
    while gap(t + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, t + step, xtol=1e-14, rtol=4 * np.finfo(float).eps,
                           maxiter=500)


def cohen_d_interval(d, df, scale, level):
    """(lower, upper): the level interval for d = t * scale."""
    t = d / scale
    return _limit(t, df, (1 + level) / 2) * scale, _limit(t, df, (1 - level) / 2) * scale


def hedges_j(df):
    """J = Γ(ν/2) / (sqrt(ν/2) Γ((ν - 1)/2)), Hedges (1981); g = J * d."""
    return math.exp(special.gammaln(df / 2) - special.gammaln((df - 1) / 2)) / math.sqrt(df / 2)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# One row per pair. A pair missing either value is dropped. The difference is the
# left column minus the right one.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
first, second = "Before training", "After training"
confidence = 95 / 100
alpha = 5 / 100
pairs = data[[first, second]].apply(pd.to_numeric, errors="coerce").dropna()
x, y = pairs[first].to_numpy(float), pairs[second].to_numpy(float)
d = x - y
n = len(d)

## Descriptive statistics

In [ ]:
descriptive = pd.DataFrame({"n": [n, n], "mean": [x.mean(), y.mean()],
                            "variance": [x.var(ddof=1), y.var(ddof=1)]}, index=[first, second])
descriptive["SD"] = np.sqrt(descriptive["variance"])
descriptive["SE"] = descriptive["SD"] / np.sqrt(n)
print(descriptive)

## The differences and effect sizes

In [ ]:
# Cohen's d (SD on N) and Cohen's d_av divide by the pooled SD of the two columns (over
# 2n and 2n - 2; d_av's is the average of the two SDs): the gain measured against the
# spread between subjects, the figure to compare with studies of two separate groups.
# Cohen's d_z divides by the SD of the differences: how consistent the gain is. None
# applies the small-sample correction, which Hedges' g_z below adds.
mean_d, sd_d = abs(d.mean()), d.std(ddof=1)
pooled_ss = (n - 1) * x.var(ddof=1) + (n - 1) * y.var(ddof=1)
effect = pd.DataFrame([[mean_d, sd_d, mean_d / np.sqrt(pooled_ss / (2 * n)),
                        mean_d / np.sqrt(pooled_ss / (2 * n - 2)), mean_d / sd_d]],
                      columns=["|mean difference|", "SD", "Cohen's d (SD on N)", "Cohen's d_av",
                               "Cohen's d_z"])
print()
print(effect.to_string(index=False))

## Confidence interval for the mean difference (always two-sided)

In [ ]:
se = sd_d / np.sqrt(n)
half = stats.t.ppf(0.5 + confidence / 2, n - 1) * se
interval = [mean_d - half, mean_d + half]
print(f"\n{confidence:.0%} interval for |mean difference|: {interval[0]:.6g} to {interval[1]:.6g}")

## Paired t test and power

In [ ]:
# The paired t test is a one-sample t test on the differences, so whatever makes
# subjects differ from each other cancels. It assumes independent pairs and roughly
# normal differences. A before-and-after gain is not proof of cause: anything else
# that changed between the two measurements could explain it.
result = stats.ttest_rel(x, y)
# Power: the chance a two-sided test at alpha rejects if the true effect equals
# the one observed (non-central t with ncp = |t|). Computed from the observed effect,
# it adds nothing to the P value; plan power from an effect that matters.
critical = stats.t.ppf(1 - alpha / 2, n - 1)
power = (stats.nct.sf(critical, n - 1, abs(result.statistic))
         + stats.nct.cdf(-critical, n - 1, abs(result.statistic)))
test = pd.DataFrame([[abs(result.statistic), n - 1, result.pvalue, stars(result.pvalue), power]],
                    columns=["t", "df", "P", "", "power"])
print()
print(test.to_string(index=False))

## Effect sizes, with confidence intervals

In [ ]:
# Cohen's d_z = mean difference / SD of the differences: the paired t over sqrt(n), and
# the Cohen's d_z above with its sign. Its interval inverts the noncentral t at the
# observed t (Steiger & Fouladi 1997); Hedges' g_z = J * d_z removes d_z's small-sample
# bias, and its interval is d_z's times J. Both steps are in cohen_d_interval.py.
from cohen_d_interval import cohen_d_interval, hedges_j


def size(value):
    """Cohen's (1988) reading of |d|: 0.2 small, 0.5 medium, 0.8 large."""
    return ["negligible", "small", "medium", "large"][sum(abs(value) >= s for s in (0.2, 0.5, 0.8))]


d_z = d.mean() / sd_d
lower, upper = cohen_d_interval(d_z, n - 1, 1 / np.sqrt(n), confidence)
j = hedges_j(n - 1)
effect_sizes = pd.DataFrame(
    [[d_z, lower, upper, size(d_z)], [j * d_z, j * lower, j * upper, size(j * d_z)]],
    index=["Cohen's d_z", "Hedges' g_z"], columns=["value", "lower", "upper", "size"],
)
print()
print(effect_sizes)

## Normality of the differences: the Shapiro-Wilk test

In [ ]:
# The paired t test assumes the differences, not the two columns, are normal.
shapiro = stats.shapiro(d)
normality = [n, shapiro.statistic, shapiro.pvalue]
print(f"\nShapiro-Wilk on the differences: n = {n}, W = {shapiro.statistic:.4f}, "
      f"P = {shapiro.pvalue:.4g}")

## What the result says

Mean weekly sales rose from 12.36 to 13.39 (\$000). The paired test looks at each rep's own change. The mean paired difference is 1.03, about one thousand dollars a week, and six of the seven reps improved.

The t statistic is 3.75 and the P value is 0.009. The P value is the chance of a mean change at least this large if nothing had really changed. The 95% confidence interval, the range of values the data agree with for the true mean gain, runs from 0.36 to 1.70 (\$000). Cohen's d_z, the mean change divided by the standard deviation of the changes, is 1.42, a large effect. The Shapiro-Wilk normality test (P = 0.28) gives no warning about the changes.

So the reps sold more after the training. But a before-and-after comparison cannot prove that the course caused the gain.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.